# 面试问题：Tool-use SFT 数据怎样序列化、做 loss mask 并防止学到坏轨迹？

**回答主线。** 一条工具轨迹不是普通对话文本，而是 `user -> assistant tool_call -> tool_result -> ... -> assistant final` 的类型化事件流。训练前要验证 call/result 对应、schema、权限和终态，再由固定 chat template 序列化；通常监督 assistant 的工具选择、参数和最终回答，不监督用户输入与外部 tool result。

下面实现事件状态机、JSON Schema 子集、序列化 span、assistant-only mask、坏轨迹隔离、原子截断、分层采样与执行式指标。重点是数据合同，不调用 Agent 框架或 SFT Trainer。


In [ ]:
import hashlib, json, re
from dataclasses import dataclass
from collections import Counter
import numpy as np

# 特殊 token 只是教学词元；真实 ID 必须来自绑定版本的 tokenizer。
SPECIAL160 = {"user": "<|user|>", "assistant_call": "<|assistant_call|>", "tool_result": "<|tool_result|>", "assistant_final": "<|assistant_final|>"}
assert len(SPECIAL160) == 4
assert len(set(SPECIAL160.values())) == 4
assert all(value.startswith("<|") for value in SPECIAL160.values())


## 1. Event schema 把角色、call_id 与 payload 分开

Tool call/result 使用同一 call_id 关联；自然语言中出现类似字符串不能代替结构字段。轨迹必须从 user 开始，以 assistant final 结束，且不能出现无调用的 result。


In [ ]:
@dataclass(frozen=True)
class Event160:
    kind: str
    content: object
    call_id: str | None = None
    tool: str | None = None

def validate_trajectory160(events):
    # pending 集合确保每个 result 恰好对应一个尚未完成的 call。
    if not events or events[0].kind != "user" or events[-1].kind != "assistant_final":
        raise ValueError("bad trajectory boundary")
    pending = set()
    for event in events:
        if event.kind not in SPECIAL160:
            raise ValueError("unknown event kind")
        if event.kind == "assistant_call":
            if not event.call_id or event.call_id in pending: raise ValueError("bad call id")
            pending.add(event.call_id)
        elif event.kind == "tool_result":
            if event.call_id not in pending: raise ValueError("orphan result")
            pending.remove(event.call_id)
    if pending: raise ValueError("missing tool result")
    return True

trace160 = [Event160("user", "北京天气？"), Event160("assistant_call", {"city": "北京"}, "c1", "weather"), Event160("tool_result", {"temp": 28}, "c1", "weather"), Event160("assistant_final", "北京 28°C")]
assert validate_trajectory160(trace160)
assert trace160[1].call_id == trace160[2].call_id
assert trace160[-1].kind == "assistant_final"


## 2. Tool 参数先过 schema，再进入 gold 数据

如果训练样本包含不存在字段或错误类型，模型会把偶然容错学成接口语义。下面实现 required、additionalProperties、string/integer/enum 子集，并返回可定位错误。


In [ ]:
WEATHER_SCHEMA160 = {"type": "object", "required": ["city"], "additionalProperties": False, "properties": {"city": {"type": "string"}, "unit": {"type": "string", "enum": ["C", "F"]}}}

def validate_args160(value, schema):
    # 教学 validator 返回全部顶层错误，生产使用通过测试的标准实现。
    errors = []
    if not isinstance(value, dict): return ["$: expected object"]
    for name in schema.get("required", []):
        if name not in value: errors.append(f"$.{name}: required")
    allowed = schema.get("properties", {})
    if not schema.get("additionalProperties", True):
        errors += [f"$.{name}: additional property" for name in value if name not in allowed]
    for name, rule in allowed.items():
        if name not in value: continue
        expected = {"string": str, "integer": int}[rule["type"]]
        if not isinstance(value[name], expected): errors.append(f"$.{name}: wrong type")
        elif "enum" in rule and value[name] not in rule["enum"]: errors.append(f"$.{name}: invalid enum")
    return errors

assert validate_args160({"city": "北京", "unit": "C"}, WEATHER_SCHEMA160) == []
assert "$.city: required" in validate_args160({"unit": "C"}, WEATHER_SCHEMA160)
assert "$.x: additional property" in validate_args160({"city": "北京", "x": 1}, WEATHER_SCHEMA160)


## 3. 序列化器返回 token 与事件 span

Loss mask 若靠重新搜索文本，重复内容会定位错误。模板应在序列化时记录每个事件的半开 token span；JSON 使用 canonical key order，避免字段顺序成为虚假标签。


In [ ]:
def event_tokens160(event):
    # 结构 payload 用 canonical JSON；文本按空白和中英文块做教学分词。
    payload = json.dumps(event.content, ensure_ascii=False, sort_keys=True, separators=(",", ":")) if not isinstance(event.content, str) else event.content
    body = re.findall(r"[A-Za-z0-9_°]+|[\u4e00-\u9fff]+|[^\s]", payload)
    header = [SPECIAL160[event.kind]]
    if event.call_id: header += [f"call={event.call_id}"]
    if event.tool: header += [f"tool={event.tool}"]
    return header + body + ["<|end|>"]

def serialize160(events):
    tokens, spans = [], []
    for event in events:
        start = len(tokens); tokens.extend(event_tokens160(event))
        spans.append({"kind": event.kind, "start": start, "stop": len(tokens), "call_id": event.call_id})
    return tokens, spans

serialized160, spans160 = serialize160(trace160)
assert spans160[0]["start"] == 0 and spans160[-1]["stop"] == len(serialized160)
assert all(a["stop"] == b["start"] for a, b in zip(spans160, spans160[1:]))
assert "tool=weather" in serialized160


## 4. Assistant-only mask 监督调用与最终回答

User 和 tool result 是上下文，不作为模型要模仿的目标；assistant_call 的工具名/参数以及 assistant_final 才监督。模板 header 是否参与 loss 必须固定，本例监督整个 assistant 事件。


In [ ]:
def loss_mask160(tokens, spans, supervised_kinds=("assistant_call", "assistant_final")):
    # 直接使用序列化 span 写 mask，避免按文本内容猜测角色。
    mask = np.zeros(len(tokens), dtype=bool)
    for span in spans:
        if span["kind"] in supervised_kinds:
            mask[span["start"]:span["stop"]] = True
    return mask

mask160 = loss_mask160(serialized160, spans160)
call_span160, result_span160 = spans160[1], spans160[2]
assert mask160[call_span160["start"]:call_span160["stop"]].all()
assert not mask160[result_span160["start"]:result_span160["stop"]].any()
assert 0 < mask160.sum() < len(mask160)


## 5. 坏调用作为 rejected/repair 数据，不能混进 gold

可修复错误保留原始 invalid call、validator error 与修正版，用于偏好或 repair 训练；gold SFT 只接收通过 schema/执行验证的调用。自动修复必须可解释且不能补造敏感字段。


In [ ]:
def repair_weather160(args):
    # 只删除未知字段并规范单位大小写，不猜测缺失 city。
    repaired = {k: v for k, v in args.items() if k in WEATHER_SCHEMA160["properties"]}
    if isinstance(repaired.get("unit"), str): repaired["unit"] = repaired["unit"].upper()
    errors = validate_args160(repaired, WEATHER_SCHEMA160)
    return repaired, errors

repaired160, errors160 = repair_weather160({"city": "北京", "unit": "c", "debug": True})
unrepairable160, missing160 = repair_weather160({"unit": "c"})
assert repaired160 == {"city": "北京", "unit": "C"} and errors160 == []
assert "$.city: required" in missing160
assert "debug" not in repaired160


## 6. 截断以 call-result 原子单元为边界

从中间截掉 tool call 却留下 result，会制造协议上不可能的训练前缀。下面把 user/final 各自作为单元，把相邻 call-result 绑定为一个单元，再按 token budget 保留完整前缀。


In [ ]:
def atomic_units160(events):
    # assistant_call 与紧随其后的 matching result 必须一起出现。
    units, i = [], 0
    while i < len(events):
        if events[i].kind == "assistant_call":
            if i + 1 >= len(events) or events[i + 1].kind != "tool_result" or events[i + 1].call_id != events[i].call_id:
                raise ValueError("non-atomic tool exchange")
            units.append(events[i:i + 2]); i += 2
        else:
            units.append([events[i]]); i += 1
    return units

def truncate_units160(events, max_tokens):
    kept, used = [], 0
    for unit in atomic_units160(events):
        size = sum(len(event_tokens160(e)) for e in unit)
        if used + size > max_tokens: break
        kept.extend(unit); used += size
    return kept

call_unit_tokens160 = sum(len(event_tokens160(e)) for e in trace160[1:3])
kept160 = truncate_units160(trace160, len(event_tokens160(trace160[0])) + call_unit_tokens160)
assert [e.kind for e in kept160] == ["user", "assistant_call", "tool_result"]
assert kept160[-1].kind != "assistant_call"
assert len(atomic_units160(trace160)) == 3


## 7. 数据切分按 scenario/template/tool family 防近重复泄漏

同一任务模板只替换城市，不应跨 train/test。采样还要控制 no-tool、single-tool、multi-tool 与 error-recovery 比例，避免模型无论问题都调用工具。


In [ ]:
def split_scenario160(scenario_id, test_percent=20):
    # 稳定哈希使整个 scenario family 固定进入同一 split。
    bucket = int(hashlib.sha256(scenario_id.encode()).hexdigest()[:8], 16) % 100
    return "test" if bucket < test_percent else "train"

records160 = [
    {"scenario": "weather-city-template", "kind": "single-tool"},
    {"scenario": "calculator-arithmetic-template", "kind": "single-tool"},
    {"scenario": "smalltalk-template", "kind": "no-tool"},
    {"scenario": "travel-plan-template", "kind": "multi-tool"},
]
counts160 = Counter(r["kind"] for r in records160)
assert split_scenario160("weather-city-template") == split_scenario160("weather-city-template")
assert counts160["single-tool"] == 2
assert set(counts160) == {"single-tool", "no-tool", "multi-tool"}


## 8. Tool-use 评测拆成选择、参数、执行和最终状态

只看最终回答会隐藏错误调用被环境偶然容错；只看 tool exact match 又会惩罚等价轨迹。分层指标先定位失败，最终状态 oracle 判断任务是否完成并检查禁止副作用。


In [ ]:
def tool_eval160(gold, predicted):
    # 四层指标分别验证 tool、canonical args、环境结果和最终答案。
    selection = gold["tool"] == predicted["tool"]
    arguments = selection and json.dumps(gold["args"], sort_keys=True) == json.dumps(predicted["args"], sort_keys=True)
    execution = arguments and gold["state"] == predicted["state"]
    final = execution and gold["final"] == predicted["final"]
    return {"selection": selection, "arguments": arguments, "execution": execution, "final": final}

gold160 = {"tool": "weather", "args": {"city": "北京"}, "state": {"lookups": 1}, "final": "28°C"}
good160 = tool_eval160(gold160, dict(gold160))
bad_arg160 = tool_eval160(gold160, {**gold160, "args": {"city": "上海"}})
assert all(good160.values())
assert bad_arg160["selection"] and not bad_arg160["arguments"]
assert not bad_arg160["final"]


## 面试总结

- Tool-use SFT 样本是类型化事件流，call_id、schema、result 与终态必须先验证。
- 序列化器同步返回 event span；mask 监督 assistant call/final，不监督 user 和外部 tool result。
- 非法调用进入 rejected/repair 数据，不混入 gold；截断必须保持 call-result 原子性。
- 评测拆选择、参数、执行、最终状态，并加入 no-tool 与 recovery slice。

延伸阅读：[Toolformer](https://arxiv.org/abs/2302.04761)、[Gorilla](https://arxiv.org/abs/2305.15334)、[ToolBench](https://arxiv.org/abs/2307.16789)。
